# Обучение head-only RuBERT для multi-label классификации отзывов

Этот ноутбук обучает первый вариант модели:

```text
RuBERT / BERT-like encoder
+
замороженные веса encoder
+
Dropout
+
Linear(hidden_size → 13)
```

То есть обучается только простая классификационная голова сверху, а сама языковая модель не дообучается.

Задача: **multi-label классификация** на 13 классов. Один отзыв может относиться сразу к нескольким классам.

Данные для обучения берутся из папок:

```text
data/labeled/wb_feedbacks_llm_labeled_from_clusters
data/labeled/wb_feedbacks_llm_labeled_from_sample
```

Валидация / финальная оценка делается на:

```text
data/labeled/golden_set_by_class
```


In [ ]:
# Если запускаешь в Google Colab, сначала подключи Google Drive

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Установка зависимостей

!pip install -q -U transformers accelerate scikit-learn pandas numpy torch tqdm


In [ ]:
import ast
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    jaccard_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE


In [ ]:
# =========================
# Основные настройки
# =========================

BASE_DIR = Path("/content/drive/MyDrive/MLops_project/project")

TRAIN_DIRS = [
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_clusters",
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_sample",
]

GOLDEN_DIR = BASE_DIR / "data" / "labeled" / "golden_set_by_class"

OUTPUT_DIR = BASE_DIR / "models" / "rubert_head_only_multilabel"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = BASE_DIR / "reports" / "rubert_head_only_multilabel"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Основная модель. Для более быстрого теста можно заменить на:
# MODEL_NAME = "cointegrated/rubert-tiny2"
MODEL_NAME = "DeepPavlov/rubert-base-cased"

MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 6
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
DROPOUT = 0.2
VALID_SIZE = 0.2
PATIENCE = 2

# Нужно ли учитывать дисбаланс классов в loss
USE_POS_WEIGHT = True

# Если Colab падает по памяти, поставь BATCH_SIZE = 8.


In [ ]:
ALLOWED_LABELS = [
    "Брак / дефект товара",
    "Низкое качество материала",
    "Проблема с размером / посадкой",
    "Несоответствие описанию",
    "Несоответствие ожиданиям / эффекту",
    "Проблема с комплектацией",
    "Проблема с упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Подделка / оригинальность",
    "Положительный отзыв",
    "Нейтральный / информационный отзыв",
    "Другая проблема",
]

LABEL2ID = {label: i for i, label in enumerate(ALLOWED_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

NUM_LABELS = len(ALLOWED_LABELS)
NUM_LABELS


In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def find_text_column(df):
    candidates = [
        "text", "review", "отзыв", "feedback", "comment", "content",
        "review_text", "Текст", "Текст отзыва"
    ]
    for col in candidates:
        if col in df.columns:
            return col

    object_cols = [c for c in df.columns if df[c].dtype == "object"]
    best_col = None
    best_len = -1
    for col in object_cols:
        avg_len = df[col].dropna().astype(str).str.len().mean()
        if avg_len > best_len:
            best_len = avg_len
            best_col = col

    if best_col is None:
        raise ValueError("Не удалось найти колонку с текстом.")
    return best_col


def find_labels_column(df):
    candidates = [
        "labels", "label", "classes", "class", "llm_labels",
        "predicted_labels", "Разметка", "Классы"
    ]
    for col in candidates:
        if col in df.columns:
            return col

    label_cols = [label for label in ALLOWED_LABELS if label in df.columns]
    if label_cols:
        return None

    raise ValueError("Не удалось найти колонку с labels и отдельные class columns.")


def parse_labels(value):
    if value is None or pd.isna(value):
        return []

    if isinstance(value, list):
        raw_items = value
    else:
        s = str(value).strip()
        if not s:
            return []

        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                raw_items = parsed
            elif isinstance(parsed, dict):
                raw_items = parsed.get("labels", [])
            else:
                raw_items = [str(parsed)]
        except Exception:
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    raw_items = parsed
                elif isinstance(parsed, dict):
                    raw_items = parsed.get("labels", [])
                else:
                    raw_items = [str(parsed)]
            except Exception:
                raw_items = re.split(r"[;,|]", s)

    labels = []
    for item in raw_items:
        label = str(item).strip().strip('"').strip("'")
        if label in LABEL2ID:
            labels.append(label)

    seen = set()
    result = []
    for label in labels:
        if label not in seen:
            result.append(label)
            seen.add(label)
    return result


def labels_to_multihot(labels):
    y = np.zeros(NUM_LABELS, dtype=np.float32)
    for label in labels:
        if label in LABEL2ID:
            y[LABEL2ID[label]] = 1.0
    return y


def load_csv_file(path):
    df = pd.read_csv(path)

    text_col = find_text_column(df)
    labels_col = find_labels_column(df)

    rows = []
    for _, row in df.iterrows():
        text = normalize_text(row[text_col])
        if not text:
            continue

        if labels_col is None:
            labels = []
            for label in ALLOWED_LABELS:
                try:
                    if int(row[label]) == 1:
                        labels.append(label)
                except Exception:
                    pass
        else:
            labels = parse_labels(row[labels_col])

        if not labels:
            continue

        rows.append({
            "text": text,
            "labels": labels,
            "source_file": str(path),
        })

    return pd.DataFrame(rows)


def load_labeled_dir(directory, prefer_stacked=False):
    directory = Path(directory)
    if not directory.exists():
        print(f"Папка не найдена: {directory}")
        return pd.DataFrame(columns=["text", "labels", "source_file"])

    csv_paths = sorted(directory.glob("*.csv"))

    if prefer_stacked:
        stacked = directory / "golden_set_all_classes_stacked.csv"
        if stacked.exists():
            csv_paths = [stacked]
        else:
            csv_paths = [
                p for p in csv_paths
                if "summary" not in p.name.lower()
                and "counts" not in p.name.lower()
                and "saved_files" not in p.name.lower()
            ]
    else:
        csv_paths = [
            p for p in csv_paths
            if "summary" not in p.name.lower()
            and "counts" not in p.name.lower()
            and "saved_files" not in p.name.lower()
            and "class_status" not in p.name.lower()
            and "run_stats" not in p.name.lower()
        ]

    frames = []
    for path in csv_paths:
        try:
            part = load_csv_file(path)
            if len(part) > 0:
                frames.append(part)
            print(f"OK: {path.name}: {len(part)} строк")
        except Exception as e:
            print(f"SKIP: {path.name}: {e}")

    if not frames:
        return pd.DataFrame(columns=["text", "labels", "source_file"])

    df = pd.concat(frames, ignore_index=True)

    grouped = []
    for text, group in df.groupby("text", sort=False):
        label_set = []
        seen = set()
        for labels in group["labels"]:
            for label in labels:
                if label not in seen:
                    label_set.append(label)
                    seen.add(label)
        grouped.append({
            "text": text,
            "labels": label_set,
            "source_file": " | ".join(sorted(set(group["source_file"].astype(str)))),
        })

    return pd.DataFrame(grouped)


In [ ]:
# Загружаем train-данные из двух папок

train_frames = []
for directory in TRAIN_DIRS:
    print("=" * 120)
    print("Загрузка:", directory)
    part = load_labeled_dir(directory, prefer_stacked=False)
    print("Итого из папки:", len(part))
    train_frames.append(part)

train_df = pd.concat(train_frames, ignore_index=True)

# Объединяем дубликаты по тексту между двумя папками
dedup_rows = []
for text, group in train_df.groupby("text", sort=False):
    label_set = []
    seen = set()
    for labels in group["labels"]:
        for label in labels:
            if label not in seen:
                label_set.append(label)
                seen.add(label)
    dedup_rows.append({
        "text": text,
        "labels": label_set,
        "source_file": " | ".join(sorted(set(group["source_file"].astype(str)))),
    })

train_df = pd.DataFrame(dedup_rows)
train_df["y"] = train_df["labels"].apply(labels_to_multihot)

print("Train before golden leakage removal:", len(train_df))
train_df.head()


In [ ]:
# Загружаем golden set

golden_df = load_labeled_dir(GOLDEN_DIR, prefer_stacked=True)
golden_df["y"] = golden_df["labels"].apply(labels_to_multihot)

print("Golden rows:", len(golden_df))
golden_df.head()


In [ ]:
# Убираем из train все тексты, которые есть в golden set, чтобы не было утечки

golden_texts = set(golden_df["text"].astype(str))
before = len(train_df)
train_df = train_df[~train_df["text"].astype(str).isin(golden_texts)].reset_index(drop=True)
after = len(train_df)

print(f"Удалено из train из-за совпадения с golden: {before - after}")
print("Train after leakage removal:", len(train_df))


In [ ]:
def class_counts(df, y_col="y"):
    if len(df) == 0:
        return pd.DataFrame({"label": ALLOWED_LABELS, "count": [0] * NUM_LABELS})

    Y = np.stack(df[y_col].values)
    counts = Y.sum(axis=0).astype(int)
    return pd.DataFrame({
        "label": ALLOWED_LABELS,
        "count": counts,
        "share": counts / len(df),
    })


train_counts = class_counts(train_df)
golden_counts = class_counts(golden_df)

display(train_counts)
display(golden_counts)

train_counts.to_csv(REPORTS_DIR / "train_class_counts.csv", index=False)
golden_counts.to_csv(REPORTS_DIR / "golden_class_counts.csv", index=False)


In [ ]:
# Делим train на train/validation.
# Golden set остается отдельно для финальной оценки.

train_part, val_part = train_test_split(
    train_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    shuffle=True,
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Train part:", len(train_part))
print("Val part:", len(val_part))
print("Golden:", len(golden_df))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ReviewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts = df["text"].astype(str).tolist()
        self.labels = np.stack(df["y"].values).astype(np.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float32),
        }


train_dataset = ReviewsDataset(train_part, tokenizer, MAX_LENGTH)
val_dataset = ReviewsDataset(val_part, tokenizer, MAX_LENGTH)
golden_dataset = ReviewsDataset(golden_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
golden_loader = DataLoader(golden_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class BertHeadOnlyMultiLabelClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_labels),
        )

        # Head-only режим: замораживаем BERT/RuBERT encoder
        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits


model = BertHeadOnlyMultiLabelClassifier(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total params:", total_params)
print("Trainable params:", trainable_params)
print("Trainable share:", trainable_params / total_params)


In [ ]:
# pos_weight для учета дисбаланса классов

Y_train = np.stack(train_part["y"].values)
positive_counts = Y_train.sum(axis=0)
negative_counts = len(Y_train) - positive_counts

pos_weight_np = negative_counts / np.clip(positive_counts, 1, None)
pos_weight_np = np.clip(pos_weight_np, 1.0, 50.0)

pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32).to(DEVICE)

if USE_POS_WEIGHT:
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
else:
    criterion = nn.BCEWithLogitsLoss()

pd.DataFrame({
    "label": ALLOWED_LABELS,
    "positive_count": positive_counts.astype(int),
    "negative_count": negative_counts.astype(int),
    "pos_weight": pos_weight_np,
})


In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = EPOCHS * len(train_loader)
num_warmup_steps = int(0.1 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)


In [ ]:
def predict_proba(model, loader):
    model.eval()
    all_probs = []
    all_targets = []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            probs = torch.sigmoid(logits)

            total_loss += loss.item() * input_ids.size(0)
            all_probs.append(probs.cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    y_prob = np.vstack(all_probs)
    y_true = np.vstack(all_targets)
    avg_loss = total_loss / len(loader.dataset)

    return y_true, y_prob, avg_loss


def binarize_with_thresholds(y_prob, thresholds):
    return (y_prob >= thresholds.reshape(1, -1)).astype(int)


def compute_metrics(y_true, y_prob, thresholds=None, prefix=""):
    if thresholds is None:
        thresholds = np.full(NUM_LABELS, 0.5)

    y_pred = binarize_with_thresholds(y_prob, thresholds)

    metrics = {
        f"{prefix}micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        f"{prefix}samples_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        f"{prefix}micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        f"{prefix}hamming_loss": hamming_loss(y_true, y_pred),
        f"{prefix}jaccard_micro": jaccard_score(y_true, y_pred, average="micro", zero_division=0),
        f"{prefix}jaccard_macro": jaccard_score(y_true, y_pred, average="macro", zero_division=0),
    }

    try:
        metrics[f"{prefix}roc_auc_micro"] = roc_auc_score(y_true, y_prob, average="micro")
    except Exception:
        metrics[f"{prefix}roc_auc_micro"] = np.nan

    try:
        metrics[f"{prefix}roc_auc_macro"] = roc_auc_score(y_true, y_prob, average="macro")
    except Exception:
        metrics[f"{prefix}roc_auc_macro"] = np.nan

    try:
        metrics[f"{prefix}pr_auc_micro"] = average_precision_score(y_true, y_prob, average="micro")
    except Exception:
        metrics[f"{prefix}pr_auc_micro"] = np.nan

    try:
        metrics[f"{prefix}pr_auc_macro"] = average_precision_score(y_true, y_prob, average="macro")
    except Exception:
        metrics[f"{prefix}pr_auc_macro"] = np.nan

    return metrics


def per_class_metrics_df(y_true, y_prob, thresholds, split_name):
    y_pred = binarize_with_thresholds(y_prob, thresholds)

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        average=None,
        zero_division=0,
    )

    rows = []
    for i, label in enumerate(ALLOWED_LABELS):
        try:
            roc_auc = roc_auc_score(y_true[:, i], y_prob[:, i])
        except Exception:
            roc_auc = np.nan

        try:
            pr_auc = average_precision_score(y_true[:, i], y_prob[:, i])
        except Exception:
            pr_auc = np.nan

        rows.append({
            "split": split_name,
            "label": label,
            "threshold": thresholds[i],
            "support": int(support[i]),
            "precision": precision[i],
            "recall": recall[i],
            "f1": f1[i],
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
        })

    return pd.DataFrame(rows)


def tune_thresholds(y_true, y_prob):
    thresholds = np.zeros(NUM_LABELS, dtype=np.float32)
    rows = []
    grid = np.arange(0.05, 0.96, 0.05)

    for i, label in enumerate(ALLOWED_LABELS):
        best_thr = 0.5
        best_f1 = -1.0

        for thr in grid:
            pred_i = (y_prob[:, i] >= thr).astype(int)
            f1_i = f1_score(y_true[:, i], pred_i, zero_division=0)

            if f1_i > best_f1:
                best_f1 = f1_i
                best_thr = float(thr)

        thresholds[i] = best_thr
        rows.append({
            "label": label,
            "best_threshold": best_thr,
            "best_val_f1": best_f1,
            "support_val": int(y_true[:, i].sum()),
        })

    return thresholds, pd.DataFrame(rows)


In [ ]:
# Обучение

history = []
best_val_macro_f1 = -1.0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        train_loss += loss.item() * input_ids.size(0)
        pbar.set_postfix({"loss": loss.item()})

    train_loss = train_loss / len(train_loader.dataset)

    y_val_true, y_val_prob, val_loss = predict_proba(model, val_loader)
    val_metrics_05 = compute_metrics(y_val_true, y_val_prob, thresholds=None, prefix="val_")

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        **val_metrics_05,
    }
    history.append(row)

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_macro_f1@0.5={val_metrics_05['val_macro_f1']:.4f}, "
        f"val_micro_f1@0.5={val_metrics_05['val_micro_f1']:.4f}"
    )

    current_score = val_metrics_05["val_macro_f1"]
    if current_score > best_val_macro_f1:
        best_val_macro_f1 = current_score
        best_state = {
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_val_macro_f1": best_val_macro_f1,
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping на эпохе {epoch}")
        break

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(REPORTS_DIR / "training_history.csv", index=False)

if best_state is not None:
    model.load_state_dict(best_state["model_state_dict"])
    print("Loaded best model from epoch:", best_state["epoch"])


In [ ]:
# Подбираем пороги по validation set

y_val_true, y_val_prob, val_loss = predict_proba(model, val_loader)

thresholds, thresholds_df = tune_thresholds(y_val_true, y_val_prob)
display(thresholds_df)

thresholds_df.to_csv(REPORTS_DIR / "thresholds_by_class.csv", index=False)

with open(REPORTS_DIR / "thresholds.json", "w", encoding="utf-8") as f:
    json.dump(
        {label: float(thresholds[i]) for i, label in enumerate(ALLOWED_LABELS)},
        f,
        ensure_ascii=False,
        indent=2,
    )


In [ ]:
# Метрики на validation после подбора порогов

val_metrics_tuned = compute_metrics(y_val_true, y_val_prob, thresholds=thresholds, prefix="val_tuned_")
val_per_class = per_class_metrics_df(y_val_true, y_val_prob, thresholds, split_name="validation")

display(pd.DataFrame([val_metrics_tuned]))
display(val_per_class)

pd.DataFrame([val_metrics_tuned]).to_csv(REPORTS_DIR / "validation_metrics_tuned.csv", index=False)
val_per_class.to_csv(REPORTS_DIR / "validation_per_class_metrics.csv", index=False)


In [ ]:
# Финальная оценка на golden set

y_gold_true, y_gold_prob, gold_loss = predict_proba(model, golden_loader)

gold_metrics_05 = compute_metrics(y_gold_true, y_gold_prob, thresholds=None, prefix="golden_05_")
gold_metrics_tuned = compute_metrics(y_gold_true, y_gold_prob, thresholds=thresholds, prefix="golden_tuned_")

gold_per_class_05 = per_class_metrics_df(
    y_gold_true,
    y_gold_prob,
    np.full(NUM_LABELS, 0.5),
    split_name="golden_threshold_0_5",
)

gold_per_class_tuned = per_class_metrics_df(
    y_gold_true,
    y_gold_prob,
    thresholds,
    split_name="golden_tuned_thresholds",
)

summary_metrics = {
    "golden_loss": gold_loss,
    **gold_metrics_05,
    **gold_metrics_tuned,
}

display(pd.DataFrame([summary_metrics]))
display(gold_per_class_tuned)

pd.DataFrame([summary_metrics]).to_csv(REPORTS_DIR / "golden_metrics_summary.csv", index=False)
gold_per_class_05.to_csv(REPORTS_DIR / "golden_per_class_metrics_threshold_0_5.csv", index=False)
gold_per_class_tuned.to_csv(REPORTS_DIR / "golden_per_class_metrics_tuned_thresholds.csv", index=False)


In [ ]:
# Сохраняем предсказания на golden set для ручного анализа ошибок

gold_pred_tuned = binarize_with_thresholds(y_gold_prob, thresholds)

gold_rows = []
for idx, row in golden_df.reset_index(drop=True).iterrows():
    true_labels = row["labels"]
    pred_labels = [
        ALLOWED_LABELS[i]
        for i in range(NUM_LABELS)
        if gold_pred_tuned[idx, i] == 1
    ]

    probs_dict = {
        ALLOWED_LABELS[i]: float(y_gold_prob[idx, i])
        for i in range(NUM_LABELS)
    }

    gold_rows.append({
        "text": row["text"],
        "true_labels": json.dumps(true_labels, ensure_ascii=False),
        "pred_labels": json.dumps(pred_labels, ensure_ascii=False),
        "is_exact_match": set(true_labels) == set(pred_labels),
        "probs_json": json.dumps(probs_dict, ensure_ascii=False),
    })

gold_predictions_df = pd.DataFrame(gold_rows)
display(gold_predictions_df.head())

gold_predictions_df.to_csv(REPORTS_DIR / "golden_predictions.csv", index=False)


In [ ]:
# Сохраняем модель, tokenizer и конфиги

MODEL_SAVE_DIR = OUTPUT_DIR / "model"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "num_labels": NUM_LABELS,
        "allowed_labels": ALLOWED_LABELS,
        "dropout": DROPOUT,
        "max_length": MAX_LENGTH,
        "thresholds": thresholds.tolist(),
    },
    MODEL_SAVE_DIR / "rubert_head_only_multilabel.pt",
)

tokenizer.save_pretrained(MODEL_SAVE_DIR / "tokenizer")

with open(MODEL_SAVE_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "allowed_labels": ALLOWED_LABELS,
            "label2id": LABEL2ID,
            "id2label": ID2LABEL,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(MODEL_SAVE_DIR / "thresholds.json", "w", encoding="utf-8") as f:
    json.dump(
        {label: float(thresholds[i]) for i, label in enumerate(ALLOWED_LABELS)},
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Модель сохранена в:", MODEL_SAVE_DIR)
print("Отчеты сохранены в:", REPORTS_DIR)


In [ ]:
# Пример инференса

def predict_labels(text, model, tokenizer, thresholds):
    model.eval()
    encoded = tokenizer(
        normalize_text(text),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(DEVICE)
    attention_mask = encoded["attention_mask"].to(DEVICE)

    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    pred_labels = [
        ALLOWED_LABELS[i]
        for i, prob in enumerate(probs)
        if prob >= thresholds[i]
    ]

    return pred_labels, {
        ALLOWED_LABELS[i]: float(probs[i])
        for i in range(NUM_LABELS)
    }


example_text = "Пришла мятая коробка, сам товар вроде целый, но упаковка ужасная."
pred_labels, probs = predict_labels(example_text, model, tokenizer, thresholds)

print("Текст:", example_text)
print("Предсказанные классы:", pred_labels)
pd.DataFrame({"label": ALLOWED_LABELS, "probability": [probs[l] for l in ALLOWED_LABELS], "threshold": thresholds})


## Что смотреть после запуска

Основные файлы с результатами появятся здесь:

```text
/content/drive/MyDrive/MLops_project/project/reports/rubert_head_only_multilabel
```

Главные файлы:

```text
training_history.csv
validation_metrics_tuned.csv
validation_per_class_metrics.csv
golden_metrics_summary.csv
golden_per_class_metrics_tuned_thresholds.csv
golden_predictions.csv
thresholds_by_class.csv
```

Сохраненная модель:

```text
/content/drive/MyDrive/MLops_project/project/models/rubert_head_only_multilabel/model/rubert_head_only_multilabel.pt
```

Главная метрика для сравнения — `golden_tuned_macro_f1`, потому что она показывает среднее качество по классам с учетом редких классов.
